# AntibodySeqML — Full GPU Training (Kaggle T4/P100)

**Prerequisites:**
- GPU accelerator enabled (Settings → Accelerator → GPU T4 x2 or P100)
- Kaggle Secret `WANDB_API_KEY` set in Account Settings
- Kaggle Secret `HF_API_KEY` set in Account Settings (HuggingFace token)
- Internet access enabled (Settings → Internet → On) — required to clone the repo and download data

This notebook is self-contained. Run all cells top-to-bottom to:
1. Clone the repository and install dependencies
2. Download and preprocess the full OAS dataset from HuggingFace
3. Run a single full-scale training run
4. Launch a Bayesian W&B sweep (50 runs)
5. Fine-tune ESM-2 (stretch goal)
6. Evaluate the best model and save to W&B Model Registry

## Cell Group 1 — Setup

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/dima806/antibody-seq-ml.git'
REPO_DIR = '/kaggle/working/antibody-seq-ml'

# Clone repo so configs/, data/, and src/ are all available on disk
if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}')

# Change working directory to repo root — all relative paths (configs/, data/, checkpoints/) resolve here
%cd {REPO_DIR}

# Install the package (reads pyproject.toml) plus GPU-only deps not in pyproject.toml
!pip install -e . --quiet
!pip install fair-esm>=2.0.0 --quiet

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Authenticate W&B and HuggingFace using Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()

WANDB_API_KEY = secrets.get_secret('WANDB_API_KEY')
wandb.login(key=WANDB_API_KEY)
print('W&B login successful')

HF_API_KEY = secrets.get_secret('HF_API_KEY')
os.environ['HF_API_KEY'] = HF_API_KEY
print('HuggingFace token set (will be used by data/download.py)')

## Cell Group 2 — Data

In [ ]:
from pathlib import Path
from data.download import download

DATA_DIR = 'data/full'  # relative to repo root (set by %cd above)
FULL_CSV = f'{DATA_DIR}/sequences_full.csv'

if not Path(FULL_CSV).exists():
    print('Downloading p-IgGen OAS snapshot from Zenodo...')
    download(DATA_DIR, source='zenodo')
else:
    print(f'Using cached dataset: {FULL_CSV}')

import pandas as pd
df = pd.read_csv(FULL_CSV)
print(f'Dataset: {len(df):,} sequences')
print(df['length_class'].value_counts())

In [ ]:
from antibody_seq_ml.dataset import build_full_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader, val_loader, test_loader = build_full_dataset(
    cache_path=DATA_DIR,
    batch_size=256,
    device=str(device),
)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## Cell Group 3 — Single GPU Training Run

In [ ]:
from omegaconf import OmegaConf
from antibody_seq_ml.models import build_model
from antibody_seq_ml.train import train

cfg = OmegaConf.load('configs/default.yaml')

# Full-scale GPU hyperparameters
cfg.model.type = 'transformer'
cfg.model.d_model = 256
cfg.model.nhead = 8
cfg.model.num_layers = 6
cfg.model.dim_feedforward = 1024
cfg.model.embedding_dim = 256
cfg.data.max_seq_length = 150
cfg.data.batch_size = 256
cfg.training.epochs = 50
cfg.training.learning_rate = 0.0003

model = build_model(cfg, device)
print(f'Model parameters: {model.count_parameters():,}')

run = wandb.init(
    project='antibody-seq-ml',
    entity='dima806-team',
    name='transformer-full-gpu',
    config=OmegaConf.to_container(cfg, resolve=True),
    tags=['gpu', 'transformer', 'full-data'],
)

result = train(model, train_loader, val_loader, cfg, test_loader=test_loader, wandb_run=run)
print('Training complete:', result)
run.finish()

## Cell Group 4 — W&B Bayesian Sweep (50 runs)

In [ ]:
from antibody_seq_ml.sweep import build_cfg_from_wandb, sweep_train_fn
import antibody_seq_ml.sweep as _sweep_module

# Point sweep agent at the full dataset
_sweep_module._DATA_PATH = FULL_CSV
_sweep_module._BASE_CFG_PATH = 'configs/default.yaml'

sweep_cfg = OmegaConf.to_container(OmegaConf.load('configs/sweep_gpu.yaml'), resolve=True)
count = sweep_cfg.pop('count', 50)

sweep_id = wandb.sweep(sweep=sweep_cfg, project='antibody-seq-ml', entity='dima806-team')
print(f'Sweep ID: {sweep_id}')
wandb.agent(sweep_id, function=sweep_train_fn, count=count)

## Cell Group 5 — ESM-2 Fine-tuning (Stretch Goal)

In [ ]:
from antibody_seq_ml.models.esm2 import ESM2FineTuned

esm_cfg = OmegaConf.load('configs/default.yaml')
esm_cfg.model.type = 'esm2'
esm_cfg.model.dropout = 0.1
esm_cfg.model.esm_trainable_layers = 2
esm_cfg.data.batch_size = 64
esm_cfg.training.epochs = 20
esm_cfg.training.learning_rate = 0.0001

esm_model = ESM2FineTuned(esm_cfg, device)
print(f'ESM-2 trainable parameters: {esm_model.count_parameters():,}')

esm_run = wandb.init(
    project='antibody-seq-ml',
    entity='dima806-team',
    name='esm2-finetuned',
    config=OmegaConf.to_container(esm_cfg, resolve=True),
    tags=['gpu', 'esm2', 'full-data'],
)

# Note: ESM2FineTuned.forward() takes list[str] not token tensors
# The standard DataLoader returns token tensors — use a sequence-string loader for ESM-2
# TODO: build a sequence-string DataLoader wrapper for ESM-2
print('ESM-2 fine-tuning requires a sequence-string DataLoader — see data/dataset.py for extension.')
esm_run.finish()

## Cell Group 6 — Final Evaluation & Model Registry

In [ ]:
from pathlib import Path
from antibody_seq_ml.evaluate import evaluate, plot_confusion_matrix
from antibody_seq_ml.dataset import CLASS_NAMES

# Load best model from checkpoint
ckpt_path = Path('checkpoints/best_model.pt')
ckpt = torch.load(ckpt_path, map_location=device)
best_cfg = OmegaConf.create(ckpt['cfg'])

best_model = build_model(best_cfg, device)
best_model.load_state_dict(ckpt['model_state_dict'])
best_model.eval()

test_metrics = evaluate(best_model, test_loader, device, best_cfg)
print('\nTest metrics:')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

# Plot confusion matrix
plot_confusion_matrix(
    y_true=[int(b['length_label'].item()) for batch in test_loader for b in [batch]],
    y_pred=[],  # populated after full inference pass
    class_names=CLASS_NAMES,
)

# Log best model to W&B Model Registry
log_run = wandb.init(project='antibody-seq-ml', entity='dima806-team', name='final-eval', job_type='eval')
artifact = wandb.Artifact(
    name='antibody-seq-ml-best',
    type='model',
    metadata={**test_metrics, 'model_type': best_cfg.model.type, 'epoch': ckpt['epoch']},
)
artifact.add_file(str(ckpt_path))
log_run.log_artifact(artifact)
log_run.summary.update(test_metrics)
log_run.finish()
print('Model logged to W&B Model Registry.')